# Kaggle v3 — Qwen + Gemma + math + PPO

Enable Internet, select an accelerator if available, and **Run All**. This entry point clones the public repo and calls the separate `kaggle_v3/train_box.py` modules.

The two default analysts are **Qwen2.5-3B-Instruct** and **Gemma 3 4B IT**. Each reads the same timestamped news independently. PPO receives each model's signals, their consensus/disagreement, and the numerical market features. Both language models stay frozen; PPO trains the trading policy.

For a real two-model run, supply `NEWS_JSONL` or both Alpaca secrets, review/accept Google's model terms on Hugging Face, and enable an `HF_TOKEN` Kaggle Secret with access to `google/gemma-3-4b-it`. Choose `NEWS_MODE="required"` to require news. Model access failures stop the run; they do not drop Gemma silently.

With no news source, `NEWS_MODE="auto"` starts a clearly labelled math-only baseline without loading either language model. This does not train or evaluate Gemma/Qwen. No orders are submitted.


In [ ]:
# Run All with these defaults. Optional settings can be changed before the first run.
REPO_REF = "main"              # first launch resolves and records an exact commit
REFRESH_REPO = False            # use a new CYCLE_ID when adopting code/data changes
CYCLE_ID = "cycle_001"
DEVICE = "auto"                # auto, cpu, cuda (NVIDIA), rocm (AMD), xpu (Intel)
NEWS_MODELS = "qwen,gemma"    # or an explicit single-model ablation: qwen / gemma
MAX_GPUS = 2                   # dual T4 automatically uses both GPUs
CPU_THREADS = 2
NEWS_MODE = "auto"             # auto, required, off
NEWS_JSONL = ""                # optional attached timestamped news JSONL
DATASET_DIR = ""               # optional replacement bars.json + dataset.json directory
UPDATES = 0                    # 0 = auto: CPU 16; GPU 40
CANDIDATES = 0                 # 0 = auto: CPU 1; GPU 3
MAX_NEWS_ARTICLES = 6000
RUN_FINAL_TEST = False          # train/validate first; reserve the holdout for a deliberate test
RESTORE_FROM = ""              # entire saved nvda-box directory, when starting a new session


In [ ]:
import os, sys, subprocess, signal, shutil, json
from pathlib import Path

REPO_URL = "https://github.com/Cheese-chess-bot/nvda-3b-alpaca-lab.git"
BASE = Path("/kaggle/working/nvda-box-v3") if Path("/kaggle/working").exists() else Path.cwd()/"nvda-box-v3-output"
PROJECT = BASE/"repo"


def execute(command, cwd=None):
    process = subprocess.Popen(list(map(str,command)), cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=os.name!="nt")
    try:
        for line in process.stdout:
            print(line,end="",flush=True)
        status = process.wait()
        if status:
            raise RuntimeError(f"Step failed (exit {status}); read the error above.")
    except BaseException:
        if process.poll() is None:
            if os.name=="nt":
                subprocess.run(["taskkill","/PID",str(process.pid),"/T","/F"],capture_output=True)
            else:
                os.killpg(process.pid,signal.SIGTERM)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL)
                    process.wait()
        raise


if RESTORE_FROM:
    if BASE.exists():
        raise RuntimeError("Restore target already exists; do not overwrite an active run.")
    restore=Path(RESTORE_FROM)
    if not (restore/"cycles").is_dir():
        raise RuntimeError("RESTORE_FROM must be the entire saved nvda-box-v3 directory.")
    shutil.copytree(restore,BASE)
BASE.mkdir(parents=True,exist_ok=True)
if not shutil.which("git"):
    raise RuntimeError("Git is required. Kaggle includes Git; on a local computer install it first.")
if not PROJECT.exists():
    execute(["git","clone",REPO_URL,PROJECT])
    execute(["git","checkout","--detach",REPO_REF],cwd=PROJECT)
else:
    origin=subprocess.check_output(["git","remote","get-url","origin"],cwd=PROJECT,text=True).strip()
    if origin.rstrip("/")!=REPO_URL:
        raise RuntimeError("This output directory contains a different repository.")
    changed=subprocess.check_output(["git","status","--porcelain"],cwd=PROJECT,text=True).strip()
    if changed:
        raise RuntimeError("The saved checkout has local edits. Preserve them before refreshing or training.")
    if REFRESH_REPO:
        execute(["git","fetch","origin",REPO_REF],cwd=PROJECT)
        execute(["git","checkout","--detach","FETCH_HEAD"],cwd=PROJECT)
revision=subprocess.check_output(["git","rev-parse","HEAD"],cwd=PROJECT,text=True).strip()
print("Repository revision:",revision)
print("This exact revision is retained for resume. Set REFRESH_REPO=True only for a new cycle.")
runner=PROJECT/"kaggle_v3"/"train_box.py"
if not runner.is_file():
    raise RuntimeError("This revision predates Kaggle Box. Select the current main branch in a fresh checkout.")
command=[sys.executable,runner,"--project",PROJECT,"--work-dir",BASE,
    "--cycle",CYCLE_ID,"--device",DEVICE,"--max-gpus",MAX_GPUS,"--threads",CPU_THREADS,
    "--models",NEWS_MODELS,"--news",NEWS_MODE,"--updates",UPDATES,"--candidates",CANDIDATES,
    "--max-news-articles",MAX_NEWS_ARTICLES]
if NEWS_JSONL:
    command += ["--news-jsonl",NEWS_JSONL]
if DATASET_DIR:
    command += ["--dataset-dir",DATASET_DIR]
if RUN_FINAL_TEST:
    command += ["--final-test"]
execute(command,cwd=PROJECT)
result_path=BASE/"cycles"/CYCLE_ID/"box_result.json"
result=json.loads(result_path.read_text())
print("\nTraining complete:",result["work_dir"])
print("Complete-ensemble coverage:",result["news_coverage"])
print("Per-model coverage:",result["model_coverage"])
print("Model revisions:",result["models"])
print("Selected policy validation:",result["winner"]["validation"])
print("Save Version WITH OUTPUTS to keep checkpoints and the holdout ledger.")
from IPython.display import FileLink,display
archive=BASE/"cycles"/CYCLE_ID/"kaggle-box-policy.zip"
display(FileLink(str(archive.relative_to("/kaggle/working")) if Path("/kaggle/working").exists() else str(archive)))


## Models, hardware and outputs

- NVIDIA: both model stages use NF4 quantization. Qwen uses FP16 compute; Gemma uses FP32 compute to avoid the FP16 path on T4. Each stage finishes and releases memory before the next starts. With two GPUs, articles are sharded between two workers; PPO then uses DDP.
- AMD ROCm / Intel XPU: Qwen uses standard FP16 weights, Gemma uses standard FP32 weights. These need a supported GPU, vendor driver and matching PyTorch build. Gemma's unquantized weights alone are roughly 16 GB; allow additional memory for activations and cache. These routes may not fit smaller devices.
- CPU: standard FP32 weights; inference on two language models is slow and memory intensive. Math-only PPO can run without model downloads.
- The launcher probes usable devices and retains a working installed PyTorch build. In auto mode an unusable accelerator falls back to CPU; an explicit unavailable device fails. GPU and full downloaded-model execution still require hardware validation.

Use Kaggle Secrets `HF_TOKEN` for model downloads and `APCA_API_KEY_ID` / `APCA_API_SECRET_KEY` for optional Alpaca news. Never paste credentials into notebook cells. The runner passes tokens through process environment only and never writes them to the configuration or policy bundle. Hugging Face access must already be granted; this notebook does not accept legal terms for you.

Outputs live in `nvda-box-v3/cycles/<CYCLE_ID>/`. Each analyst has its own cache under `news_cache/qwen` or `news_cache/gemma`. Both enabled analysts must produce valid outputs on enough shared articles to pass the coverage gate. `box_result.json` reports combined and per-model coverage; `run_manifest.json` records model revisions, dataset/configuration fingerprints and the selected policy for a later controller. Keep the **entire output folder** to resume; the policy ZIP alone is insufficient.

V3 has its own feature/bundle schema. Existing V2 checkpoints cannot be resumed as V3. Source, data, model-set or device changes require a new cycle. Retain evaluation history across all versions; creating a new folder does not make old test data unseen.

## Planned self-evolution

This release prepares the model stage. It does not start an automatic evolution loop. See `kaggle_v3/EVOLUTION.md` for the next-stage design: bounded challenger training, fresh chronological evaluation, fixed promotion gates, staged paper observation and rollback. V3 automatic promotion is disabled. `RUN_FINAL_TEST=False` keeps the final holdout reserved; switching it on consumes the test window through the ledger but does not deploy a policy. Bundled dates have already been inspected and cannot establish a new champion.

For future code updates, use `REFRESH_REPO=True` with a new `CYCLE_ID`. Both pretrained models can have prior knowledge of historical events; historical results are not contamination-free evidence or a profitability guarantee.

Primary sources: [Gemma 3 4B model and access terms](https://huggingface.co/google/gemma-3-4b-it), [Transformers 4.51.3 Gemma API](https://huggingface.co/docs/transformers/v4.51.3/model_doc/gemma3).
